# Controversial-Plot Snapshot — Model Comparison

A **single-time-point** comparison in the vein of `run_analysis.py` (`plot_compare`):
instead of one image, we run each model **N times per controversial chart** on the
neutral prompt *"Interpret this plot."*, score every response with **VADER** sentiment,
and compare how the models frame each topic.

**How to use**
1. Put your chart images in the `plots/` folder (created below).
2. List each one in the `TOPICS` config (label → filename).
3. Make sure `.env.local` (next to the notebook or one level up) has
   `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, and `XAI_API_KEY`.
4. Run all cells. Raw responses cache to `results/responses.csv` — set `RERUN = False`
   to re-analyse without re-calling the APIs (fully reproducible offline).

Outputs (in `results/`): `responses.csv`, `descriptives.csv`, per-topic error-bar and
word-frequency PNGs, and the headline `cross_topic_comparison.png`.


## Imports

In [ ]:
import base64, json, os, re, time
from collections import Counter
from pathlib import Path

import anthropic as anthropic_sdk
import matplotlib.pyplot as plt
import nltk
import numpy as np
import openai as openai_sdk
import pandas as pd
from scipy import stats
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

## 1. Configuration

Edit `TOPICS` to match the images you drop in `plots/`. `RERUN = False` skips the API
calls and rebuilds every table/plot from the cached `results/responses.csv`.

In [ ]:
# ── What we run ──────────────────────────────────────────────────────────────
PROMPT  = "Interpret this plot."   # same neutral prompt as plot_compare
N_RUNS  = 10                       # runs per model, per topic
DELAY_S = 0.5                      # pause between runs (rate-limit friendliness)

# True  -> call the APIs and overwrite the cache.
# False -> load results/responses.csv and just redo stats/plots (no API calls).
RERUN = True

MODELS = {
    "Claude":  "claude-sonnet-5",
    "ChatGPT": "gpt-4o",
    "Grok":    "grok-4",
}

# ── The controversial plots ──────────────────────────────────────────────────
# Drop your chart images into PLOTS_DIR and list them here: label -> filename.
TOPICS = {
    "CO2 / Climate":        "co2.png",
    "COVID deaths & masks": "covid.png",
    "Race & IQ":            "raceiq.png",
    "Gender pay gap":       "paygap.png",
}

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR  = Path.cwd()          # directory the notebook runs from
PLOTS_DIR = BASE_DIR / "plots"  # <- put your topic images here
OUT_DIR   = BASE_DIR / "results"
PLOTS_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

# ── Style ────────────────────────────────────────────────────────────────────
COLOURS = {"Claude": "#d97706", "ChatGPT": "#16a34a", "Grok": "#2563eb"}

STOP_WORDS = set(nltk_stopwords.words("english")) | {
    "shows", "show", "plot", "graph", "data", "figure", "chart", "image",
    "line", "lines", "values", "value", "number", "numbers", "time",
    "trend", "trends", "overall", "however", "also",
}

print(f"Topics : {list(TOPICS)}")
print(f"Models : {list(MODELS)}")
print(f"Plan   : {len(TOPICS)} topics x {len(MODELS)} models x {N_RUNS} runs "
      f"= {len(TOPICS) * len(MODELS) * N_RUNS} API calls")
print(f"Drop your chart images in: {PLOTS_DIR}")

## 2. Setup — env + clients

In [ ]:
def load_dotenv(p: Path):
    if not p.exists():
        return
    for line in p.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            os.environ.setdefault(k.strip(), v.strip())

# .env.local next to the notebook, then one level up (matches plot_compare layout)
for cand in (BASE_DIR / ".env.local", BASE_DIR.parent / ".env.local"):
    load_dotenv(cand)

anthropic_client = anthropic_sdk.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
openai_client    = openai_sdk.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
xai_client       = openai_sdk.OpenAI(
    api_key  = os.environ.get("XAI_API_KEY") or os.environ.get("AI_API_KEY", ""),
    base_url = "https://api.x.ai/v1",
)
vader = SentimentIntensityAnalyzer()
print("Clients ready.")

## 3. API callers

In [ ]:
def call_claude(b64, mt):
    r = anthropic_client.messages.create(
        model=MODELS["Claude"], max_tokens=1024, temperature=1,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": mt, "data": b64}},
            {"type": "text",  "text": PROMPT},
        ]}],
    )
    return next((b.text for b in r.content if b.type == "text"), "")

def call_compat(client, model, b64, mt):
    r = client.chat.completions.create(
        model=model, max_tokens=1024, temperature=1,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:{mt};base64,{b64}"}},
            {"type": "text",      "text": PROMPT},
        ]}],
    )
    return r.choices[0].message.content or ""

CALLERS = {
    "Claude":  lambda b64, mt: call_claude(b64, mt),
    "ChatGPT": lambda b64, mt: call_compat(openai_client, MODELS["ChatGPT"], b64, mt),
    "Grok":    lambda b64, mt: call_compat(xai_client,    MODELS["Grok"],    b64, mt),
}

def encode_image(path: Path):
    mt = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
    return base64.b64encode(path.read_bytes()).decode(), mt

## 4. Run the trials (or load the cache)

Loops every topic × model × run, VADER-scores each response, and stores everything in a
long-format DataFrame `df` (one row per call). Missing images are skipped with a warning;
API errors record a `None` score so one bad call never aborts the run.

In [ ]:
RESPONSES_CSV = OUT_DIR / "responses.csv"

def run_all():
    rows = []
    for topic, fname in TOPICS.items():
        img = PLOTS_DIR / fname
        if not img.exists():
            print(f"!! SKIP {topic}: image not found -> {img}")
            continue
        b64, mt = encode_image(img)
        print(f"\n=== {topic}  ({fname}) ===")
        for i in range(1, N_RUNS + 1):
            print(f"  Run {i:>2}/{N_RUNS}", end="  ", flush=True)
            for name, caller in CALLERS.items():
                try:
                    text = caller(b64, mt)
                    s = vader.polarity_scores(text)["compound"]
                    print(f"{name}={s:+.3f}", end="  ", flush=True)
                except Exception as e:
                    text, s = "", None
                    print(f"{name}=ERR({type(e).__name__})", end="  ", flush=True)
                rows.append({"topic": topic, "run": i, "model": name,
                             "score": s, "response": text})
            print()
            if i < N_RUNS:
                time.sleep(DELAY_S)
    return pd.DataFrame(rows)

if RERUN or not RESPONSES_CSV.exists():
    df = run_all()
    if df.empty:
        raise SystemExit("No results — did you add images to the plots/ folder?")
    df.to_csv(RESPONSES_CSV, index=False)
    print(f"\nSaved raw results -> {RESPONSES_CSV}")
else:
    df = pd.read_csv(RESPONSES_CSV)
    print(f"Loaded cached results <- {RESPONSES_CSV}  ({len(df)} rows)")

df.head()

## 5. Descriptive statistics (per topic × model)

In [ ]:
records = []
for (topic, model), g in df.groupby(["topic", "model"], sort=False):
    v = g["score"].dropna().to_numpy(dtype=float)
    records.append({
        "topic": topic, "model": model,
        "n": len(v),
        "mean":   v.mean()          if len(v)     else np.nan,
        "sd":     v.std(ddof=1)     if len(v) > 1 else 0.0,
        "median": np.median(v)      if len(v)     else np.nan,
        "min":    v.min()           if len(v)     else np.nan,
        "max":    v.max()           if len(v)     else np.nan,
    })

desc = pd.DataFrame(records)
desc.to_csv(OUT_DIR / "descriptives.csv", index=False)
print("Saved: descriptives.csv")
desc

## 6. Significance tests (per topic)

Same battery as `plot_compare`: Kruskal-Wallis across the three models, then pairwise
Mann-Whitney U with a Bonferroni correction and a rank-biserial effect size `r`.
Non-parametric because VADER compound scores are bounded and rarely normal.

In [ ]:
def sig_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

def tests_for_topic(sub):
    groups = {}
    for m in MODELS:
        v = sub[sub.model == m]["score"].dropna().to_numpy(dtype=float)
        if len(v):
            groups[m] = v
    out = {"n_models": len(groups)}
    if len(groups) >= 2:
        H, p_kw = stats.kruskal(*groups.values())
        out["kruskal_H"], out["kruskal_p"] = H, p_kw
        names = list(groups)
        pairs = [(a, b) for i, a in enumerate(names) for b in names[i + 1:]]
        pairwise = {}
        for a, b in pairs:
            U, p = stats.mannwhitneyu(groups[a], groups[b], alternative="two-sided")
            p_bonf = min(p * len(pairs), 1.0)
            eff = 1 - (2 * U) / (len(groups[a]) * len(groups[b]))
            pairwise[f"{a} vs {b}"] = dict(U=U, p_raw=p, p_bonf=p_bonf, r=eff)
        out["pairwise"] = pairwise
    return out

all_tests = {}
for topic in TOPICS:
    sub = df[df.topic == topic]
    if sub.empty:
        continue
    t = tests_for_topic(sub)
    all_tests[topic] = t
    print(f"\n=== {topic} ===")
    if t["n_models"] < 2:
        print("  (need >=2 models with data)")
        continue
    print(f"  Kruskal-Wallis: H={t['kruskal_H']:.3f}  p={t['kruskal_p']:.4f}  {sig_stars(t['kruskal_p'])}")
    for pair, r in t["pairwise"].items():
        print(f"    {pair:<20} p_bonf={r['p_bonf']:.4f} {sig_stars(r['p_bonf'])}  r={r['r']:+.3f}")

(OUT_DIR / "stats.json").write_text(json.dumps(all_tests, indent=2, default=float))
print("\nSaved: stats.json")

## 7. Per-topic sentiment error bars  (one chart per controversial plot)

In [ ]:
def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

def plot_errorbars_topic(topic, sub_desc, out_path):
    sub_desc = sub_desc.set_index("model").reindex([m for m in MODELS if m in sub_desc["model"].values])
    names  = sub_desc.index.tolist()
    means  = sub_desc["mean"].tolist()
    sds    = sub_desc["sd"].tolist()
    colors = [COLOURS.get(m, "#888888") for m in names]
    n0     = int(sub_desc["n"].iloc[0]) if len(sub_desc) else 0

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(names, means, yerr=sds, capsize=8, color=colors,
                  alpha=0.85, error_kw=dict(elinewidth=1.5, ecolor="black"))
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
    ax.set_ylabel("VADER Compound Score")
    ax.set_title(f"{topic}\nMean Sentiment ± 1 SD  (n={n0} runs each)")
    ax.set_ylim(-1.1, 1.1)
    for bar, mean, sd in zip(bars, means, sds):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + sd + 0.04,
                f"{mean:+.3f}", ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    print(f"Saved: {out_path.name}")
    plt.show()

for topic in TOPICS:
    sub_desc = desc[desc.topic == topic]
    if sub_desc.empty:
        continue
    plot_errorbars_topic(topic, sub_desc, OUT_DIR / f"errorbars_{slugify(topic)}.png")

## 8. Headline: cross-topic model comparison

Grouped bars — topics on the x-axis, one coloured bar per model (mean ± 1 SD). This is the
single-time-point summary: *does a given model systematically read these charts more
positively/negatively than the others, and does that flip by topic?*

In [ ]:
topics = [t for t in TOPICS if not desc[desc.topic == t].empty]
models = list(MODELS)
x = np.arange(len(topics))
width = 0.8 / max(len(models), 1)
offsets = [(-(len(models) - 1) / 2 + i) * width for i in range(len(models))]

fig, ax = plt.subplots(figsize=(max(8, 2.4 * len(topics)), 5.5))
for i, m in enumerate(models):
    means, sds = [], []
    for t in topics:
        row = desc[(desc.topic == t) & (desc.model == m)]
        means.append(row["mean"].iloc[0] if len(row) else np.nan)
        sds.append(row["sd"].iloc[0] if len(row) else 0.0)
    ax.bar(x + offsets[i], means, width, yerr=sds, capsize=5, label=m,
           color=COLOURS.get(m, "#888888"), alpha=0.85,
           error_kw=dict(elinewidth=1.2, ecolor="black"))

ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(topics, rotation=15, ha="right")
ax.set_ylabel("VADER Compound Score")
ax.set_title(f"Sentiment by Model across Controversial Plots  (mean ± 1 SD, n={N_RUNS})")
ax.set_ylim(-1.1, 1.1)
ax.legend(title="Model")
fig.tight_layout()
fig.savefig(OUT_DIR / "cross_topic_comparison.png", dpi=150)
print("Saved: cross_topic_comparison.png")
plt.show()

## 9. Word frequency per topic  (what language does each model reach for?)

Top words across all runs of a topic, split by model. Sentiment says *how* positive;
this says *which* framing words drive it.

In [ ]:
def plot_word_freq_topic(topic, sub, out_path, top_n=15):
    all_words = re.findall(r"[a-z]+", " ".join(str(t) for t in sub["response"]).lower())
    top_words = [w for w, _ in Counter(
        w for w in all_words if w not in STOP_WORDS and len(w) > 2
    ).most_common(top_n)]
    if not top_words:
        print(f"  (no words for {topic})")
        return
    models = [m for m in MODELS if (sub.model == m).any()]
    x = np.arange(len(top_words))
    width = 0.8 / max(len(models), 1)
    offsets = [(-(len(models) - 1) / 2 + i) * width for i in range(len(models))]

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, m in enumerate(models):
        words = re.findall(r"[a-z]+",
                           " ".join(str(t) for t in sub[sub.model == m]["response"]).lower())
        c = Counter(w for w in words if w not in STOP_WORDS and len(w) > 2)
        ax.bar(x + offsets[i], [c.get(w, 0) for w in top_words], width,
               label=m, color=COLOURS.get(m, "#888888"), alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(top_words, rotation=40, ha="right", fontsize=9)
    ax.set_ylabel(f"Frequency (across {N_RUNS} runs)")
    ax.set_title(f"{topic}: Top {top_n} Words by Model  (stop words removed)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    print(f"Saved: {out_path.name}")
    plt.show()

for topic in TOPICS:
    sub = df[df.topic == topic]
    if sub.empty:
        continue
    plot_word_freq_topic(topic, sub, OUT_DIR / f"words_{slugify(topic)}.png")

## Notes & caveats

- **VADER is a lexicon tuned for social media.** On measured, hedged model prose it reads
  broad tone, not stance — treat the compound score as "warmth of framing", not a fact-check.
- **`temperature=1`** gives real run-to-run variation; that spread is the point of N runs.
- **Reproducibility:** the exact model *outputs* aren't deterministic, but every run is cached
  to `results/responses.csv`. Commit that CSV and set `RERUN = False` to reproduce all tables
  and figures identically without spending API calls.
- **Adding topics:** drop an image in `plots/`, add a `label -> filename` line to `TOPICS`,
  re-run. Everything downstream (stats, both plot families) picks it up automatically.
